# 13 — Performance: CPU vs MPS (Apple GPU)

PyTorch lets us flip a single string (`device='cpu'` ↔ `'mps'`) and run the **same model code** on the Apple-silicon GPU. The interesting question is *when* this is actually faster: small kernels are dominated by launch latency, and the CPU often wins on lensing problems with `npix < a few hundred`.

This notebook quantifies the trade-off by benchmarking three representative workloads:

1. **Forward pass** — a Sérsic profile evaluated on a 2-D grid;
2. **Backward pass** — same Sérsic + a dummy MSE loss + autograd;
3. **End-to-end fit** — Adam optimization for ``N`` epochs.

Each measurement uses `lensing.benchmarks.compare_devices`, which wraps `time.perf_counter` with proper device synchronization so we capture *all* in-flight kernel launches (a pitfall every PyTorch benchmark must avoid).

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
device, dtype = gl.config.setup(seed=42, device="cpu")


## 0. Environment overview

Before running any timing we record what hardware we're on. Apple's MPS backend is fast on M1/M2/M3 chips for moderate-size kernels but loses on tiny ones because of the cost of issuing a GPU command buffer.

In [ ]:
print('PyTorch :', torch.__version__)
print('Devices :', gl.benchmarks.available_devices())
# The MPS allocator currently has no peak-memory query, so we
# only print sizes when running on CUDA.
if torch.cuda.is_available():
    print('CUDA dev:', torch.cuda.get_device_name(0))


## 1. Forward-pass scaling with image size

We sweep ``npix ∈ {32, 64, 128, 256, 512}`` and record the wall time of one Sérsic forward pass on each device. The expectation is that for small images the CPU wins (kernel launch ~ 50–100 µs on Apple Silicon), and the GPU starts winning once each kernel has enough work to amortise the launch overhead — typically around ~256² pixels for elementwise ops, lower for matmul-heavy ones.

In [ ]:
import pandas as pd
sizes = [32, 64, 128, 256, 512]
forward_results = []
for n in sizes:
    df = gl.benchmarks.compare_devices(
        gl.benchmarks.sersic_forward_workload(npix=n, batch=1),
        n_warmup=3, n_repeats=10,
    )
    df['npix'] = n
    forward_results.append(df.reset_index())
forward = pd.concat(forward_results, ignore_index=True)
forward


In [ ]:
# Plot wall-time vs grid size, one line per device.
fig, ax = plt.subplots(figsize=(8, 5))
for dev, sub in forward.groupby('device'):
    ax.errorbar(sub['npix'], sub['mean_ms'], yerr=sub['std_ms'],
                marker='o', label=dev, capsize=3)
ax.set(xscale='log', yscale='log',
       xlabel='image side (pixels)',
       ylabel='forward wall time [ms]',
       title='Sérsic forward pass — per-device wall time')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.show()


**Reading the chart**: where the two lines cross is the *break-even point* of the workload. At image sizes above the crossing, MPS is preferable; below it, the CPU's lower kernel launch latency wins. For a Sérsic profile, the cross-over on this hardware is typically between ``npix=128`` and ``npix=256`` — well above the typical 64×64 postage stamps used in the weak-lensing notebook 05, so for that workload **CPU is the right default**.

## 2. Backward-pass timing

Autodiff doubles roughly the FLOP count (forward + reverse), and adds an allocation pass for the gradient buffers. Backward tends to make the GPU more competitive because the extra kernels share the same memory allocations.

In [ ]:
backward_results = []
for n in sizes:
    df = gl.benchmarks.compare_devices(
        gl.benchmarks.sersic_backward_workload(npix=n),
        n_warmup=3, n_repeats=8,
    )
    df['npix'] = n
    backward_results.append(df.reset_index())
backward = pd.concat(backward_results, ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
for dev, sub in backward.groupby('device'):
    ax.errorbar(sub['npix'], sub['mean_ms'], yerr=sub['std_ms'],
                marker='s', label=dev, capsize=3)
ax.set(xscale='log', yscale='log',
       xlabel='image side (pixels)',
       ylabel='forward+backward wall time [ms]',
       title='Sérsic forward + backward — per-device wall time')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.show()
backward


## 3. End-to-end fit timing

The most realistic benchmark: how long does a complete Sérsic fit take on each device? We run a fixed-budget Adam loop (500 iterations) and record the wall time. This includes the scheduler, the constraint enforcement, and parameter updates — *not* just the forward kernels.

In [ ]:
from time import perf_counter

def end_to_end_fit(device, npix=128, epochs=500):
    xy = gl.data.coordinate_grid(npix=npix, deltapix=0.05).to(device)
    truth = gl.light.Sersic(Ie=5., Re=1., n=4., x0=0., y0=0.,
                             e1=0.1, e2=-0.05).to(device)
    with torch.no_grad():
        image = truth(xy)
    model = gl.light.Sersic(Ie=2., Re=1.5, n=2.5, x0=0., y0=0.,
                             e1=0., e2=0.).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    # Sync before timing.
    gl.benchmarks.synchronize(device)
    t0 = perf_counter()
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(xy)
        loss = ((pred - image) ** 2).mean()
        loss.backward()
        opt.step()
    gl.benchmarks.synchronize(device)
    return perf_counter() - t0

rows = []
for dev in gl.benchmarks.available_devices():
    # Warm-up to absorb kernel-compilation / cache-population costs.
    end_to_end_fit(dev, npix=128, epochs=20)
    t = end_to_end_fit(dev, npix=128, epochs=500)
    rows.append({'device': dev, 'wall_s': t,
                 'epoch_ms': t * 1000 / 500})
fit_df = pd.DataFrame(rows).set_index('device')
fit_df['speedup_vs_cpu'] = fit_df.loc['cpu', 'wall_s'] / fit_df['wall_s']
fit_df


**Discussion**: in our experience on Apple-silicon laptops, MPS pulls ahead of the CPU only when the per-iteration kernel work is large enough — typically ``npix ≳ 256`` for elementwise Sérsic ops, lower for matmul-heavy workloads such as the U-Net training in notebook 12. Two practical recommendations:

* For **single-image fits at npix ≤ 128**, keep `device='cpu'` (you save the host↔device transfer too).
* For **batched fits / training loops** where you have many images at once, use `device='mps'` — the GPU's parallelism amortises the launch latency across the batch.

## 4. Where MPS shines: U-Net training

Convolutional networks are matmul-heavy and benefit much more from the GPU. We time one epoch of the U-Net from notebook 12 on each device for a head-to-head comparison.

In [ ]:
from torch.utils.data import DataLoader

def time_unet_epoch(device):
    ds = gl.ml.datasets.LensSourcePairDataset(
        n_samples=64, npix=48, deltapix=0.05, seed=0)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    model = gl.ml.models.UNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    # Warm-up
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = ((model(x) - y) ** 2).mean()
        loss.backward(); opt.step()
        break
    gl.benchmarks.synchronize(device)
    t0 = perf_counter()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = ((model(x) - y) ** 2).mean()
        loss.backward(); opt.step()
    gl.benchmarks.synchronize(device)
    return perf_counter() - t0

rows = []
for dev in gl.benchmarks.available_devices():
    rows.append({'device': dev, 'epoch_s': time_unet_epoch(dev)})
unet_df = pd.DataFrame(rows).set_index('device')
if 'cpu' in unet_df.index:
    unet_df['speedup_vs_cpu'] = unet_df.loc['cpu', 'epoch_s'] / unet_df['epoch_s']
unet_df


Conv-heavy training is exactly the regime PyTorch's MPS path is tuned for, and we expect a ≥2× speed-up on M1/M2 vs. CPU for the U-Net (depending on macOS version and PyTorch build).

**Caveat**: PyTorch's MPS backend was first released in v1.12 (May 2022) and is still maturing — a handful of operators do not yet have an MPS kernel and silently fall back to CPU. Always check both backends agree before trusting a result.